# 01 · Validate analytics (v2, revised)

Deterministic analytics for the Gallup engagement agent: fit the fixed models, run **every** trap check on **every** finding, build verified payloads, and only then let the language model explain them.

**What changed in this revision**

1. **No hardcoded trap answers.** The old version applied CAUTION through masks for Q11, Q07 and a built-in `Q01 < 3` control. Every diagnostic now runs on every reportable finding, so the same code works on real Gallup data where nobody knows where the traps are.
2. **All eight traps are covered:** prior-outcome check (X2), within-team check (X3), threshold discovery (X4), subgroup check (X5), direction-flip check (X6), nonresponse caveat (X7), small-group suppression (X8); X1 is handled by the department control.
3. **Clearer statuses:** `PASSED_CHECKS` (was `ROBUST`), `WITHIN_GROUP` (report the within-department result), `CAUTION` (do not prioritize).
4. **Extra stability check:** each finding is refit without extreme responders.
5. **Verifiable calculation IDs:** the ID hashes the data, the code, the controls and the result, and every payload is written to `reports/calculation_log.jsonl`.
6. **Caveat text is built from the health report numbers**, so nothing goes stale when the data changes.
7. **Odds are translated into probabilities by Python**, never by the language model.
8. **Scorecard against the answer key** (graders only) fills the Pending cells in Tables 9 and 10.
9. **Language model:** cleaned prompt for nontechnical business users, low reasoning effort, and an automatic check of every answer (causal words, calculation ID, numbers traceable to the payload).
10. **Portable paths:** no `/Users/...` paths; set environment variables to override.

In [1]:
# PROCESS 1 – SETUP AND CONFIGURATION

from pathlib import Path
import os
import sys
import json
import hashlib
import datetime as dt

import numpy as np
import pandas as pd

# Portable paths: the notebook lives in <APP_ROOT>/notebooks.
# Override any of these with environment variables if your layout differs.
NOTEBOOK_DIR = Path.cwd()
APP_ROOT = Path(os.getenv(
    "GALLUP_APP_ROOT",
    NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR,
))
WORKSPACE = APP_ROOT.parent
DATASET = os.getenv("GALLUP_DATASET", "gallup_v2_5K")

DATA_DIR = APP_ROOT / "data" / "dev" / DATASET
DATA_PATH = DATA_DIR / "agent_data"
REPORTS_DIR = DATA_DIR / "reports"

SYNTH_ROOT = WORKSPACE / "Business-Practicum-Group-10-" / "synthetic_data_v2"
REFERENCE_CODE = Path(os.getenv("GALLUP_REFERENCE_CODE", SYNTH_ROOT / "gallup_v2_code"))
ANSWER_KEY_PATH = Path(os.getenv(
    "GALLUP_ANSWER_KEY",
    SYNTH_ROOT / f"{DATASET}_answer_key" / "ground_truth.json",
))

for p in [DATA_PATH, REPORTS_DIR, REFERENCE_CODE]:
    assert p.exists(), f"Missing path: {p} (set the matching GALLUP_* environment variable)"

sys.path.insert(0, str(REFERENCE_CODE))
import stats_tools as st

# Every threshold the pipeline uses lives here, so it is reported and versioned.
CONFIG = {
    "fdr_q": 0.05,                    # Benjamini-Hochberg false discovery rate
    "min_group_size": 5,              # confidentiality threshold
    "meaningful_ratio": [0.91, 1.10], # OR / IRR outside this band is meaningful
    "meaningful_beta": 0.05,          # |standardized beta| for performance
    "low_score_threshold": 3,         # threshold test: item score below 3
    "threshold_min_ratio": 1.25,      # below-3 group must differ by >= 25% in odds or rate
    "threshold_min_beta": 0.10,       # ... or by 0.10 SD of performance
    "new_hire_years": 2,              # subgroup check: tenure under 2 years
    "subgroup_min_log_ratio": 0.20,   # new hires vs others must differ by this much (log scale)
    "subgroup_min_beta": 0.05,
    "prior_reduction_flag": 0.50,     # X2: effect shrinks >= 50% with prior rating
    "team_shrinkage_flag": 0.50,      # X3: effect shrinks >= 50% within teams ...
    "team_within_p": 0.05,            # ... and is not significant within teams
    "extreme_item_share": 0.80,       # extreme responder: >80% of answers are 1 or 5
    "stability_max_change": 0.25,     # effect may move at most 25% without extreme responders
}

ITEMS = list(st.ITEMS)

ITEM_LABELS = {
    "Q01": "Clear expectations",
    "Q02": "Materials and equipment",
    "Q03": "Opportunity to do best",
    "Q04": "Recognition",
    "Q05": "Someone cares",
    "Q06": "Development encouraged",
    "Q07": "Opinions count",
    "Q08": "Mission and purpose",
    "Q09": "Quality commitment",
    "Q10": "Best friend at work",
    "Q11": "Progress conversations",
    "Q12": "Learn and grow",
}

CONTROL_LABELS = {
    "tenure_years": "tenure",
    "department": "department",
    "role_level": "role level",
    "pay_band": "pay band",
    "shift_type": "work shift",
    "prior_performance_rating": "prior performance",
}

RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

print("App root:", APP_ROOT)
print("Data path:", DATA_PATH)
print("Reference code:", REFERENCE_CODE)
print("Run ID:", RUN_ID)

App root: /Users/selvianiaulyah/Documents/AI-Agent/gallup-engagement-agent
Data path: /Users/selvianiaulyah/Documents/AI-Agent/gallup-engagement-agent/data/dev/gallup_v2_5K/agent_data
Reference code: /Users/selvianiaulyah/Documents/AI-Agent/Business-Practicum-Group-10-/synthetic_data_v2/gallup_v2_code
Run ID: 20261010T170027Z


In [2]:
# PROCESS 2 – LOAD AGENT-VISIBLE DATA AND FINGERPRINT IT

survey = pd.read_csv(DATA_PATH / "survey_responses.csv")
indiv = pd.read_csv(DATA_PATH / "individual_outcomes.csv")
team = pd.read_csv(DATA_PATH / "team_outcomes.csv")


def file_hash(paths):
    h = hashlib.sha256()
    for p in paths:
        h.update(Path(p).read_bytes())
    return h.hexdigest()[:12]


DATA_FINGERPRINT = file_hash(sorted(DATA_PATH.glob("*.csv")))
CODE_FINGERPRINT = hashlib.sha256(
    (REFERENCE_CODE / "stats_tools.py").read_bytes()
    + json.dumps(CONFIG, sort_keys=True).encode()
).hexdigest()[:12]

print("Survey:", survey.shape)
print("Individual outcomes:", indiv.shape)
print("Team outcomes:", team.shape)
print("Data fingerprint:", DATA_FINGERPRINT)
print("Code + config fingerprint:", CODE_FINGERPRINT)

Survey: (10000, 22)
Individual outcomes: (10000, 9)
Team outcomes: (800, 4)
Data fingerprint: 1318216791ec
Code + config fingerprint: 3269447f39ac


In [3]:
# PROCESS 3 – BUILD ANALYTICAL SAMPLE

df = survey.merge(indiv, on=["wave", "employee_id"])

w1_all = df[df["wave"] == 1].copy()
w1 = w1_all[w1_all["responded"] == 1].copy()

# Flags used by the diagnostics
w1["new_hire"] = (w1["tenure_years"] < CONFIG["new_hire_years"]).astype(float)
extreme_share = w1[ITEMS].isin([1, 5]).mean(axis=1)
w1["extreme_responder"] = extreme_share > CONFIG["extreme_item_share"]

# Below-threshold indicators for every item (threshold discovery, X4)
for item in ITEMS:
    w1[f"{item}_low"] = (w1[item] < CONFIG["low_score_threshold"]).astype(float)

print("Wave 1 employees:", len(w1_all))
print("Wave 1 respondents:", len(w1))
print(f"Survey response rate: {w1_all['responded'].mean():.1%}")
print(f"New hires (< {CONFIG['new_hire_years']} years): {w1['new_hire'].mean():.1%}")
print(f"Extreme responders: {w1['extreme_responder'].mean():.1%}")

Wave 1 employees: 5000
Wave 1 respondents: 4024
Survey response rate: 80.5%
New hires (< 2 years): 32.7%
Extreme responders: 7.8%


## Model specifications

One fixed setup per outcome (Matrix A). Unlike the earlier version, the absence model has **no built-in `Q01 < 3` term**: thresholds are discovered from the data in Process 5 and added only if they pass the test.

In [4]:
# PROCESS 4 – MODEL SPECIFICATIONS AND GENERIC FITTING

OUTCOMES = {
    "turnover": {
        "y": "left_voluntarily_12m",
        "family": "logit",
        "effect_type": "Odds Ratio",
        "method": "Logistic Regression",
        "numeric": ["tenure_years"],
        "categorical": ["department", "role_level", "pay_band"],
    },
    "absence": {
        "y": "unplanned_absence_days_6m",
        "family": "negbin",
        "effect_type": "Incidence Rate Ratio",
        "method": "Negative Binomial Regression",
        "numeric": ["tenure_years"],
        "categorical": ["department", "shift_type"],
    },
    "performance": {
        "y": "performance_rating",
        "family": "ols_z",
        "effect_type": "Standardized Beta",
        "method": "Standardized Linear Regression (approximates the ordinal model)",
        "numeric": ["prior_performance_rating", "tenure_years"],
        "categorical": ["department"],
    },
}


def is_ratio(outcome):
    return OUTCOMES[outcome]["family"] in ("logit", "negbin")


def prepare(d, outcome):
    """Performance runs on z-scores (items, rating, prior rating); others use raw scores."""
    d = d.copy()
    if OUTCOMES[outcome]["family"] == "ols_z":
        for c in ITEMS + ["performance_rating", "prior_performance_rating"]:
            d[c] = st.zscore(d[c])
    return d


def fit_outcome(d, outcome, extra_numeric=(), interactions=(), drop_numeric=(), cluster=True):
    spec = OUTCOMES[outcome]
    frame = prepare(d, outcome)
    numeric = ITEMS + list(extra_numeric) + [c for c in spec["numeric"] if c not in drop_numeric]
    X = st.design(frame, numeric, spec["categorical"], interactions)
    y = frame[spec["y"]]
    groups = frame["team_id"] if cluster else None
    if spec["family"] == "logit":
        return st.logit(y, X, groups)
    if spec["family"] == "negbin":
        return st.negbin(y, X, groups)
    return st.ols(y, X, groups)


def get_effect(fit, term, outcome):
    """Returns (effect, ci_low, ci_high, p_value) on the reporting scale."""
    if is_ratio(outcome):
        e, lo, hi = fit.ratio(term)
    else:
        e = float(fit.params[term])
        lo, hi = fit.ci(term)
    return float(e), float(lo), float(hi), float(fit.pvalues[term])


def log_effect(effect, outcome):
    """Effect on an additive scale (log ratio or beta) for comparisons."""
    return float(np.log(effect)) if is_ratio(outcome) else float(effect)


base_fits = {o: fit_outcome(w1, o) for o in OUTCOMES}
for o, f in base_fits.items():
    print(f"{o:12s} n = {f.n}")

turnover     n = 4024
absence      n = 4024
performance  n = 4024


In [5]:
# PROCESS 5 – THRESHOLD DISCOVERY FOR EVERY ITEM AND OUTCOME (X4)
# Adds "item below 3" to the model one item at a time and keeps the
# thresholds that survive FDR correction and are large enough to matter.

threshold_rows = []
for outcome in OUTCOMES:
    for item in ITEMS:
        term = f"{item}_low"
        if w1[term].sum() < CONFIG["min_group_size"]:
            continue
        fit = fit_outcome(w1, outcome, extra_numeric=[term])
        e, lo, hi, p = get_effect(fit, term, outcome)
        threshold_rows.append({
            "item": item, "term": term, "outcome": outcome,
            "effect": e, "ci_low": lo, "ci_high": hi, "p_value": p,
            "share_below": float(w1[term].mean()),
        })

thresholds = pd.DataFrame(threshold_rows)
thresholds["fdr_significant"] = st.bh_fdr(thresholds["p_value"].to_numpy(), q=CONFIG["fdr_q"])


def threshold_meaningful(row):
    if is_ratio(row["outcome"]):
        r = CONFIG["threshold_min_ratio"]
        return row["effect"] >= r or row["effect"] <= 1 / r
    return abs(row["effect"]) >= CONFIG["threshold_min_beta"]


thresholds["meaningful"] = thresholds.apply(threshold_meaningful, axis=1)
thresholds["discovered"] = thresholds["fdr_significant"] & thresholds["meaningful"]

THRESHOLD_TERMS = {
    o: thresholds.loc[(thresholds["outcome"] == o) & thresholds["discovered"], "term"].tolist()
    for o in OUTCOMES
}

print("Threshold tests run:", len(thresholds))
print("Discovered thresholds:", THRESHOLD_TERMS)
thresholds[thresholds["discovered"]].round(4)

Threshold tests run: 36
Discovered thresholds: {'turnover': [], 'absence': ['Q01_low'], 'performance': []}


,item,term,outcome,effect,ci_low,ci_high,p_value,share_below,fdr_significant,meaningful,discovered
12,Q01,Q01_low,absence,1.8051,1.5926,2.046,0.0,0.0999,True,True,True


In [6]:
# PROCESS 6 – FINAL MODELS (discovered thresholds included as data-driven terms)

fits = {
    o: fit_outcome(w1, o, extra_numeric=THRESHOLD_TERMS[o])
    for o in OUTCOMES
}

for o, f in fits.items():
    print(f"{o:12s} n = {f.n}  extra terms = {THRESHOLD_TERMS[o] or 'none'}")

turnover     n = 4024  extra terms = none
absence      n = 4024  extra terms = ['Q01_low']
performance  n = 4024  extra terms = none


In [7]:
# PROCESS 7 – ALL TESTS: 36 ITEM-OUTCOME EFFECTS PLUS DISCOVERED THRESHOLDS

rows = []
for outcome, fit in fits.items():
    terms = [(i, i, "linear") for i in ITEMS]
    terms += [(t.replace("_low", ""), t, "threshold") for t in THRESHOLD_TERMS[outcome]]
    for item, term, kind in terms:
        e, lo, hi, p = get_effect(fit, term, outcome)
        rows.append({
            "item": item, "term": term, "finding_type": kind, "outcome": outcome,
            "effect_type": OUTCOMES[outcome]["effect_type"],
            "effect": e, "ci_low": lo, "ci_high": hi, "p_value": p, "n": fit.n,
        })

results = pd.DataFrame(rows)
results["fdr_significant"] = st.bh_fdr(results["p_value"].to_numpy(), q=CONFIG["fdr_q"])


def is_meaningful(row):
    if row["finding_type"] == "threshold":
        return threshold_meaningful(row)
    if is_ratio(row["outcome"]):
        low, high = CONFIG["meaningful_ratio"]
        return row["effect"] < low or row["effect"] > high
    return abs(row["effect"]) >= CONFIG["meaningful_beta"]


results["meaningful_effect"] = results.apply(is_meaningful, axis=1)
results["reportable"] = results["fdr_significant"] & results["meaningful_effect"]

print("Tests:", len(results), "| reportable:", int(results["reportable"].sum()))
results[results["reportable"]].sort_values(["outcome", "effect"]).round(4)

Tests: 37 | reportable: 9


,item,term,finding_type,outcome,effect_type,effect,ci_low,ci_high,p_value,n,fdr_significant,meaningful_effect,reportable
18,Q07,Q07,linear,absence,Incidence Rate Ratio,0.8124,0.7828,0.8431,0.0000,4024,True,True,True
16,Q05,Q05,linear,absence,Incidence Rate Ratio,0.8763,0.8426,0.9114,0.0000,4024,True,True,True
24,Q01,Q01_low,threshold,absence,Incidence Rate Ratio,1.8051,1.5926,2.0460,0.0000,4024,True,True,True
35,Q11,Q11,linear,performance,Standardized Beta,0.0753,0.0458,0.1049,0.0000,4024,True,True,True
31,Q07,Q07,linear,performance,Standardized Beta,0.0967,0.0706,0.1227,0.0000,4024,True,True,True
25,Q01,Q01,linear,performance,Standardized Beta,0.1261,0.0959,0.1563,0.0000,4024,True,True,True
3,Q04,Q04,linear,turnover,Odds Ratio,0.6913,0.6143,0.7779,0.0000,4024,True,True,True
11,Q12,Q12,linear,turnover,Odds Ratio,0.8396,0.7518,0.9376,0.0019,4024,True,True,True
4,Q05,Q05,linear,turnover,Odds Ratio,0.8413,0.7511,0.9423,0.0028,4024,True,True,True


## Diagnostics: every check on every reportable finding

| Check | Trap | Runs on | Flag rule |
|---|---|---|---|
| Prior-outcome sensitivity | X2 | performance findings | effect shrinks ≥ 50% when prior rating is added |
| Within-team comparison | X3 | all findings | effect shrinks ≥ 50% within teams **and** is not significant there |
| Direction flip | X6 | all findings | company-wide and within-department directions differ |
| Subgroup (new hires) | X5 | linear findings | interaction survives FDR **and** the separate estimates for new hires and others differ enough |
| Stability without extreme responders | data quality | all findings | effect moves > 25% or loses significance |

In [8]:
# PROCESS 8 – GENERIC DIAGNOSTIC FUNCTIONS


def linear_y(d, outcome):
    """Outcome on a linear scale for the fixed-effects checks."""
    spec = OUTCOMES[outcome]
    if spec["family"] == "logit":
        return d[spec["y"]].astype(float)               # linear probability model
    if spec["family"] == "negbin":
        return np.log1p(d[spec["y"]].astype(float))     # log(1 + days)
    return pd.Series(st.zscore(d[spec["y"]]), index=d.index)


def prior_outcome_check(term, outcome):
    """X2: how much of the effect disappears once prior performance is controlled."""
    if outcome != "performance" or "prior_performance_rating" not in OUTCOMES[outcome]["numeric"]:
        return {"applies": False}
    extra = THRESHOLD_TERMS[outcome]
    with_prior = get_effect(fits[outcome], term, outcome)[0]
    without = fit_outcome(w1, outcome, extra_numeric=extra, drop_numeric=["prior_performance_rating"])
    without_prior = get_effect(without, term, outcome)[0]
    reduction = 1 - abs(log_effect(with_prior, outcome) / log_effect(without_prior, outcome))
    return {
        "applies": True,
        "effect_without_prior": round(without_prior, 4),
        "effect_with_prior": round(with_prior, 4),
        "reduction": round(reduction, 3),
        "flag": reduction >= CONFIG["prior_reduction_flag"],
    }


def team_confounding_check(term, outcome):
    """X3: compare the department-controlled estimate with a within-team (team fixed effects) estimate."""
    spec = OUTCOMES[outcome]
    frame = prepare(w1, outcome)
    y = linear_y(frame, outcome)
    numeric = ITEMS + THRESHOLD_TERMS[outcome] + spec["numeric"]
    other_cats = [c for c in spec["categorical"] if c != "department"]

    X_dept = st.design(frame, numeric, spec["categorical"])
    dept_fit = st.ols(y, X_dept, frame["team_id"])

    X_team = st.design(frame, numeric, other_cats).drop(columns="Intercept")
    X_team = X_team - X_team.groupby(frame["team_id"]).transform("mean")
    y_team = y - y.groupby(frame["team_id"]).transform("mean")
    team_fit = st.ols(y_team, X_team, frame["team_id"])

    b_dept = float(dept_fit.params[term])
    b_team = float(team_fit.params[term])
    lo, hi = team_fit.ci(term)
    p_team = float(team_fit.pvalues[term])
    shrinkage = 1 - abs(b_team / b_dept) if b_dept != 0 else 0.0
    return {
        "linear_scale": {"logit": "probability", "negbin": "log(1 + days)", "ols_z": "z-score"}[spec["family"]],
        "dept_controlled": round(b_dept, 4),
        "within_team": round(b_team, 4),
        "within_team_ci": [round(float(lo), 4), round(float(hi), 4)],
        "within_team_p": round(p_team, 4),
        "shrinkage": round(shrinkage, 3),
        "flag": shrinkage >= CONFIG["team_shrinkage_flag"] and p_team > CONFIG["team_within_p"],
    }


def direction_flip_check(term, outcome):
    """X6: does the company-wide (pooled) direction differ from the within-department direction?"""
    frame = prepare(w1, outcome)
    y = linear_y(frame, outcome)
    pooled = st.ols(y, st.design(frame, [term]), frame["team_id"])
    within = st.ols(y, st.design(frame, [term], ["department"]), frame["team_id"])
    b_pooled, b_within = float(pooled.params[term]), float(within.params[term])
    flip = np.sign(b_pooled) != np.sign(b_within) and float(within.pvalues[term]) < 0.05
    return {
        "pooled": round(b_pooled, 4),
        "within_department": round(b_within, 4),
        "flag": bool(flip),
    }


def subgroup_check(term, outcome):
    """X5: does the effect differ for new hires? Pooled interaction test plus stratified estimates."""
    extra = THRESHOLD_TERMS[outcome] + ["new_hire"]
    inter = fit_outcome(w1, outcome, extra_numeric=extra, interactions=[(term, "new_hire")])
    name = f"{term}:new_hire"
    b_int = float(inter.params[name])
    p_int = float(inter.pvalues[name])
    strata = {}
    for label, mask in [("new_hires", w1["new_hire"] == 1), ("others", w1["new_hire"] == 0)]:
        f = fit_outcome(w1[mask], outcome, extra_numeric=THRESHOLD_TERMS[outcome])
        e, lo, hi, p = get_effect(f, term, outcome)
        strata[label] = {"effect": round(e, 4), "ci": [round(lo, 4), round(hi, 4)], "n": int(f.n)}
    # The groups must really differ: a pooled interaction alone can borrow another item's
    # interaction because the Q12 items move together.
    min_size = CONFIG["subgroup_min_log_ratio"] if is_ratio(outcome) else CONFIG["subgroup_min_beta"]
    diff = log_effect(strata["new_hires"]["effect"], outcome) - log_effect(strata["others"]["effect"], outcome)
    large_enough = abs(diff) >= min_size and np.sign(diff) == np.sign(b_int)
    return {"interaction": round(b_int, 4), "p_value": p_int, "strata_difference": round(float(diff), 4),
            "large_enough": bool(large_enough), **strata}


def stability_check(term, outcome, base_effect):
    """Refit without extreme responders; flag if the effect moves too much or loses significance."""
    sub = w1[~w1["extreme_responder"]]
    f = fit_outcome(sub, outcome, extra_numeric=THRESHOLD_TERMS[outcome])
    e, lo, hi, p = get_effect(f, term, outcome)
    change = abs(log_effect(e, outcome) - log_effect(base_effect, outcome)) / abs(log_effect(base_effect, outcome))
    return {
        "effect_without_extreme": round(e, 4),
        "relative_change": round(change, 3),
        "p_value": p,
        "flag": change > CONFIG["stability_max_change"] or p > 0.05,
    }


print("Diagnostic functions ready.")

Diagnostic functions ready.


In [9]:
# PROCESS 9 – RUN ALL DIAGNOSTICS ON EVERY REPORTABLE FINDING

findings = results[results["reportable"]].copy().reset_index(drop=True)
diagnostics = {}

for _, r in findings.iterrows():
    key = (r["term"], r["outcome"])
    d = {
        "prior_outcome": prior_outcome_check(r["term"], r["outcome"]),
        "team_confounding": team_confounding_check(r["term"], r["outcome"]),
        "direction_flip": direction_flip_check(r["term"], r["outcome"]),
        "stability": stability_check(r["term"], r["outcome"], r["effect"]),
    }
    if r["finding_type"] == "linear":
        d["subgroup"] = subgroup_check(r["term"], r["outcome"])
    diagnostics[key] = d

# FDR across the subgroup tests, so the subgroup check does not create false alarms
sub_keys = [k for k, d in diagnostics.items() if "subgroup" in d]
if sub_keys:
    sub_pass = st.bh_fdr(np.array([diagnostics[k]["subgroup"]["p_value"] for k in sub_keys]), q=CONFIG["fdr_q"])
    for k, ok in zip(sub_keys, sub_pass):
        s = diagnostics[k]["subgroup"]
        s["flag"] = bool(ok and s["large_enough"])

summary = pd.DataFrame([
    {
        "term": k[0], "outcome": k[1],
        "prior_flag": d["prior_outcome"].get("flag", False),
        "prior_reduction": d["prior_outcome"].get("reduction"),
        "team_flag": d["team_confounding"]["flag"],
        "team_shrinkage": d["team_confounding"]["shrinkage"],
        "flip_flag": d["direction_flip"]["flag"],
        "subgroup_flag": d.get("subgroup", {}).get("flag", False),
        "unstable_flag": d["stability"]["flag"],
    }
    for k, d in diagnostics.items()
])
summary

,term,outcome,prior_flag,prior_reduction,team_flag,team_shrinkage,flip_flag,subgroup_flag,unstable_flag
0,Q04,turnover,False,NaN,False,0.036,False,True,False
1,Q05,turnover,False,NaN,False,0.026,False,False,False
2,Q12,turnover,False,NaN,False,-0.029,False,False,False
3,Q05,absence,False,NaN,False,-0.118,False,False,False
4,Q07,absence,False,NaN,True,0.938,False,False,False
5,Q01_low,absence,False,NaN,False,0.073,False,False,False
6,Q01,performance,False,-0.162,False,-0.031,False,False,False
7,Q07,performance,False,-0.117,False,0.151,True,False,False
8,Q11,performance,True,0.742,False,0.104,False,False,False


In [10]:
# PROCESS 10 – CLASSIFY EACH FINDING FROM ITS DIAGNOSTICS (no item-specific rules)

STATUS_LABELS = {
    "PASSED_CHECKS": "passed all of the agent's checks for common analysis traps",
    "WITHIN_GROUP": "holds within departments, although the company-wide pattern points the other way",
    "CAUTION": "did not pass the agent's checks, so it should not be prioritized",
}


def classify(r, d):
    reasons, notes = [], []
    status, eligible = "PASSED_CHECKS", True

    po = d["prior_outcome"]
    if po.get("flag"):
        status, eligible = "CAUTION", False
        reasons.append(
            f"The effect shrank by {po['reduction']:.0%} once prior performance was taken into account "
            f"({po['effect_without_prior']:.2f} to {po['effect_with_prior']:.2f}), so the score may reflect "
            "past performance rather than explain future performance."
        )

    tc = d["team_confounding"]
    if tc["flag"]:
        status, eligible = "CAUTION", False
        reasons.append(
            f"Comparing employees within the same team, the association shrank by {tc['shrinkage']:.0%} "
            f"and is no longer distinguishable from zero (within-team estimate {tc['within_team']:+.3f} on the "
            f"{tc['linear_scale']} scale, 95% CI {tc['within_team_ci'][0]:+.3f} to {tc['within_team_ci'][1]:+.3f}). "
            "A team or manager factor likely explains much of the pattern."
        )

    stb = d["stability"]
    if stb["flag"]:
        status, eligible = "CAUTION", False
        reasons.append(
            f"Without extreme responders the effect moved by {stb['relative_change']:.0%} "
            f"(to {stb['effect_without_extreme']:.2f}), so it is not stable."
        )

    fl = d["direction_flip"]
    if fl["flag"] and status != "CAUTION":
        status = "WITHIN_GROUP"
        reasons.append(
            f"Company-wide the association points the other way ({fl['pooled']:+.3f}), but within departments "
            f"it is {fl['within_department']:+.3f}. Departments differ in how they score this item and rate "
            "performance, so the within-department result is the one to report."
        )

    sg = d.get("subgroup")
    if sg and sg.get("flag"):
        notes.append(
            f"The association differs for employees with under {CONFIG['new_hire_years']} years of tenure: "
            f"{sg['new_hires']['effect']:.2f} (95% CI {sg['new_hires']['ci'][0]:.2f} to {sg['new_hires']['ci'][1]:.2f}, "
            f"n = {sg['new_hires']['n']:,}) versus {sg['others']['effect']:.2f} "
            f"(95% CI {sg['others']['ci'][0]:.2f} to {sg['others']['ci'][1]:.2f}, n = {sg['others']['n']:,}) for others."
        )

    if r["finding_type"] == "threshold":
        notes.append(
            f"This is a threshold pattern: the association is concentrated among employees scoring below "
            f"{CONFIG['low_score_threshold']} on {ITEM_LABELS[r['item']]}, rather than a steady change per point."
        )

    return status, eligible, " ".join(reasons) or None, " ".join(notes) or None


classified = [classify(r, diagnostics[(r["term"], r["outcome"])]) for _, r in findings.iterrows()]
findings["status"] = [c[0] for c in classified]
findings["priority_eligible"] = [c[1] for c in classified]
findings["status_reason"] = [c[2] for c in classified]
findings["notes"] = [c[3] for c in classified]

final_findings = findings.sort_values(["status", "outcome", "item"]).reset_index(drop=True)

print(final_findings["status"].value_counts().to_string())
print("Priority-eligible findings:", int(final_findings["priority_eligible"].sum()))
final_findings[["item", "term", "outcome", "effect", "ci_low", "ci_high", "status", "priority_eligible"]].round(4)

status
PASSED_CHECKS    6
CAUTION          2
WITHIN_GROUP     1
Priority-eligible findings: 7


,item,term,outcome,effect,ci_low,ci_high,status,priority_eligible
0,Q07,Q07,absence,0.8124,0.7828,0.8431,CAUTION,False
1,Q11,Q11,performance,0.0753,0.0458,0.1049,CAUTION,False
2,Q01,Q01_low,absence,1.8051,1.5926,2.0460,PASSED_CHECKS,True
3,Q05,Q05,absence,0.8763,0.8426,0.9114,PASSED_CHECKS,True
4,Q01,Q01,performance,0.1261,0.0959,0.1563,PASSED_CHECKS,True
5,Q04,Q04,turnover,0.6913,0.6143,0.7779,PASSED_CHECKS,True
6,Q05,Q05,turnover,0.8413,0.7511,0.9423,PASSED_CHECKS,True
7,Q12,Q12,turnover,0.8396,0.7518,0.9376,PASSED_CHECKS,True
8,Q07,Q07,performance,0.0967,0.0706,0.1227,WITHIN_GROUP,True


In [11]:
# PROCESS 11 – SMALL-GROUP SUPPRESSION (X8)


def suppress_small_groups(data, group_col, value_cols, min_n=None):
    """Aggregate by group and blank out every group with fewer than min_n people."""
    min_n = min_n or CONFIG["min_group_size"]
    agg = data.groupby(group_col).agg(n=(value_cols[0], "size"), **{c: (c, "mean") for c in value_cols})
    agg["suppressed"] = agg["n"] < min_n
    agg.loc[agg["suppressed"], value_cols] = np.nan
    return agg.reset_index()


team_view = suppress_small_groups(w1, "team_id", ["Q04", "left_voluntarily_12m"])
team_sizes_w1 = w1_all.groupby("team_id").size()

print("Teams with < 5 employees (whole team):", int((team_sizes_w1 < CONFIG["min_group_size"]).sum()))
print("Teams suppressed in survey results (< 5 respondents):", int(team_view["suppressed"].sum()))
print("Any suppressed team showing a value:",
      bool(team_view.loc[team_view["suppressed"], ["Q04", "left_voluntarily_12m"]].notna().any().any()))

Teams with < 5 employees (whole team): 20
Teams suppressed in survey results (< 5 respondents): 20
Any suppressed team showing a value: False


In [12]:
# PROCESS 12 – HEALTH REPORT AND CAVEATS (text built from the report's own numbers)

with open(REPORTS_DIR / "health_report.json") as f:
    health_report = json.load(f)

health_summary = health_report["summary"]
health_checks = {c["code"]: c for c in health_report["checks"]}

print("Checks:", health_summary["checks_run"], "| PASS:", health_summary["pass"],
      "| WARN:", health_summary["warn"], "| FAIL:", health_summary["fail"])
print("Analysis allowed:", health_summary["analysis_allowed"])


def caveat_text(code, item=None, outcome=None):
    c = health_checks[code]
    m = c.get("metrics", {})
    if code == "NR_LEAVERS":
        return (f"The survey response rate was {m['leaver_rate']:.1%} among employees who later left versus "
                f"{m['stayer_rate']:.1%} among those who stayed, so turnover associations are likely understated.")
    if code.startswith("NR_SMD_"):
        return (f"Survey respondents and nonrespondents differed in absence (standardized difference {m['smd']:+.2f}), "
                "so nonresponse may bias absence estimates.")
    if code == "NR_GAP_department":
        return f"Survey response rates differed across departments ({c['detail']})."
    if code == "NR_GAP_pay_band":
        return f"Survey response rates differed across pay bands ({c['detail']})."
    if code == "MS_DIF":
        label = f"{ITEM_LABELS[item]} ({item})" if item else "Some items"
        return f"{label} scores differ across departments beyond overall engagement, so results are compared within departments."
    if code == "RS_EXTREME":
        return (f"{m['share']:.1%} of respondents gave mostly extreme answers (1 or 5); "
                "the result was rechecked without them.")
    if code == "AG_SIMPSON":
        pair = next((p for p in m.get("pairs", []) if p["item"] == item), None)
        if pair:
            return (f"Company-wide, {ITEM_LABELS[item]} and performance move in opposite directions "
                    f"(r = {pair['pooled_r']:+.2f}), but within departments they move together (r = {pair['within_r']:+.2f}).")
        return "The company-wide and within-department patterns differ."
    if code == "COV_SMALL_TEAMS":
        return f"{m['teams']} teams ({m['share']:.0%}) have fewer than {CONFIG['min_group_size']} members; their results are not shown."
    if code == "SV_LEAVER_ENGAGEMENT":
        return "Employees who left were less engaged, so later survey waves can look better simply because they left."
    if code == "SV_STAYER_SHARE":
        return f"Later waves mix stayers and new hires ({c['detail']}), so trends compare the same employees only."
    return c["detail"]


# Material caveats per outcome for an overall question; others are added when they apply.
PRIMARY_CAVEATS = {
    "turnover": {"NR_LEAVERS"},
    "absence": {"NR_SMD_unplanned_absence_days_6m", "MS_DIF"},
    "performance": {"MS_DIF"},
}
HANDLED_BY_MODEL = {"OL_PERF_GAP_department", "OL_TURNOVER_GAP_department", "OL_PRIOR_ANCHOR"}


def select_caveats(outcome, item=None, question_scope="overall", status=None, stability_flag=False):
    selected = []
    for code, check in health_checks.items():
        if check["status"] != "WARN" or code in HANDLED_BY_MODEL:
            continue
        affects = check.get("affects", [])
        if "all" not in affects and outcome not in affects and question_scope not in affects:
            continue
        if code == "COV_SMALL_TEAMS" and question_scope not in ("team", "group"):
            continue
        if code.startswith("SV_") and question_scope != "trend":
            continue
        if code == "MS_DIF" and item not in check.get("metrics", {}).get("items", []):
            continue
        if code == "AG_SIMPSON":
            pairs = check.get("metrics", {}).get("pairs", [])
            if not any(p["item"] == item and p["outcome"] == outcome for p in pairs):
                continue
            if status != "WITHIN_GROUP":
                continue
        if code == "RS_EXTREME" and not stability_flag:
            continue  # extreme answers checked; only material if the finding was unstable
        material = code in PRIMARY_CAVEATS.get(outcome, set()) or code in {"AG_SIMPSON", "RS_EXTREME"}
        if question_scope == "overall" and not material:
            continue
        selected.append({
            "code": code,
            "detail": caveat_text(code, item, outcome),
            "evidence_strength": check.get("evidence_strength"),
        })
    return selected


print()
print("Example (Q04, turnover):", [c["detail"] for c in select_caveats("turnover", "Q04")])

Checks: 45 | PASS: 32 | WARN: 13 | FAIL: 0
Analysis allowed: {'turnover': True, 'absence': True, 'performance': True}

Example (Q04, turnover): ['The survey response rate was 57.7% among employees who later left versus 85.6% among those who stayed, so turnover associations are likely understated.']


In [13]:
# PROCESS 13 – DETERMINISTIC BUSINESS METRICS (Python translates odds; the model never does)

BASELINES = {
    "turnover": float(w1["left_voluntarily_12m"].mean()),
    "absence": float(w1["unplanned_absence_days_6m"].mean()),
}


def business_metrics(row):
    outcome, e = row["outcome"], row["effect"]
    lo, hi = row["ci_low"], row["ci_high"]
    if outcome == "turnover":
        p0 = BASELINES["turnover"]
        odds0 = p0 / (1 - p0)
        to_p = lambda r: odds0 * r / (1 + odds0 * r)
        base = {
            "relative_odds_change_pct": round((e - 1) * 100, 1),
            "relative_odds_change_ci_pct": [round((lo - 1) * 100, 1), round((hi - 1) * 100, 1)],
            "interpretation_unit": "odds, not probability",
            "baseline_turnover_rate_pct": round(p0 * 100, 1),
        }
        if row["finding_type"] == "linear":
            base["illustration"] = {
                "turnover_rate_at_average_pct": round(p0 * 100, 1),
                "turnover_rate_one_point_higher_pct": round(to_p(e) * 100, 1),
                "note": "Illustration for an otherwise average survey respondent; computed from the odds ratio.",
            }
        return base
    if outcome == "absence":
        m0 = BASELINES["absence"]
        out = {
            "relative_rate_change_pct": round((e - 1) * 100, 1),
            "relative_rate_change_ci_pct": [round((lo - 1) * 100, 1), round((hi - 1) * 100, 1)],
            "baseline_absence_days_6m": round(m0, 2),
        }
        if row["finding_type"] == "linear":
            out["illustration"] = {
                "absence_days_at_average": round(m0, 2),
                "absence_days_one_point_higher": round(m0 * e, 2),
            }
        return out
    return {
        "standardized_beta": round(e, 3),
        "interpretation_unit": "standard deviations of performance rating per standard deviation of the item score",
    }


print(json.dumps(business_metrics(final_findings[(final_findings["term"] == "Q04")
                                                & (final_findings["outcome"] == "turnover")].iloc[0]), indent=2))

{
  "relative_odds_change_pct": -30.9,
  "relative_odds_change_ci_pct": [
    -38.6,
    -22.2
  ],
  "interpretation_unit": "odds, not probability",
  "baseline_turnover_rate_pct": 13.1,
  "illustration": {
    "turnover_rate_at_average_pct": 13.1,
    "turnover_rate_one_point_higher_pct": 9.4,
    "note": "Illustration for an otherwise average survey respondent; computed from the odds ratio."
  }
}


In [14]:
# PROCESS 14 – VERIFIABLE CALCULATION ID AND LOG

LOG_PATH = REPORTS_DIR / "calculation_log.jsonl"


def make_calculation_id(payload):
    """Hash of data + code/config + the full specification and result. Same inputs, same ID."""
    basis = json.dumps({
        "data": DATA_FINGERPRINT,
        "code": CODE_FINGERPRINT,
        "outcome": payload["outcome"],
        "term": payload["term"],
        "method": payload["method"],
        "controls": payload["controls"],
        "extra_terms": payload["extra_terms"],
        "effect": payload["effect"],
        "ci": payload["confidence_interval"],
    }, sort_keys=True)
    short = hashlib.sha256(basis.encode()).hexdigest()[:10].upper()
    return f"{payload['outcome'].upper()}-{payload['term'].upper()}-{short}"


def log_calculation(payload):
    with open(LOG_PATH, "a") as f:
        f.write(json.dumps({"run_id": RUN_ID, **payload}) + "\n")


print("Calculation log:", LOG_PATH)

Calculation log: /Users/selvianiaulyah/Documents/AI-Agent/gallup-engagement-agent/data/dev/gallup_v2_5K/reports/calculation_log.jsonl


In [15]:
# PROCESS 15 – BUILD STRUCTURED FINDINGS (one generic builder for every finding)


def text_or_none(v):
    return v if isinstance(v, str) and v.strip() else None


def build_finding_payload(row, question_scope="overall"):
    outcome = row["outcome"]
    spec = OUTCOMES[outcome]
    diag = diagnostics[(row["term"], outcome)]
    payload = {
        "dataset": DATASET,
        "data_fingerprint": DATA_FINGERPRINT,
        "code_fingerprint": CODE_FINGERPRINT,
        "item": row["item"],
        "item_label": ITEM_LABELS[row["item"]],
        "term": row["term"],
        "finding_type": row["finding_type"],
        "outcome": outcome,
        "method": spec["method"],
        "effect_type": row["effect_type"],
        "effect": round(float(row["effect"]), 4),
        "confidence_interval": {"low": round(float(row["ci_low"]), 4), "high": round(float(row["ci_high"]), 4), "level": 0.95},
        "p_value": float(row["p_value"]),
        "fdr_significant": bool(row["fdr_significant"]),
        "meaningful_effect": bool(row["meaningful_effect"]),
        "sample_size": int(row["n"]),
        "controls": spec["numeric"] + spec["categorical"],
        "also_adjusted_for": "the other 11 Q12 items",
        "extra_terms": THRESHOLD_TERMS[outcome],
        "clustered_by": "team_id",
        "status": row["status"],
        "priority_eligible": bool(row["priority_eligible"]),
        "status_reason": text_or_none(row["status_reason"]),
        "notes": text_or_none(row["notes"]),
        "diagnostics": diag,
        "business_metrics": business_metrics(row),
        "health": {
            "analysis_allowed": bool(health_summary["analysis_allowed"][outcome]),
            "caveats": select_caveats(outcome, row["item"], question_scope, row["status"], diag["stability"]["flag"]),
        },
    }
    payload["calculation_id"] = make_calculation_id(payload)
    return payload


structured_findings = [build_finding_payload(r) for _, r in final_findings.iterrows()]
for p in structured_findings:
    log_calculation(p)

print(f"Built and logged {len(structured_findings)} findings.")
pd.DataFrame([{k: p[k] for k in ["calculation_id", "item", "outcome", "effect", "status"]} for p in structured_findings])

Built and logged 9 findings.


,calculation_id,item,outcome,effect,status
0,ABSENCE-Q07-AA370B754F,Q07,absence,0.8124,CAUTION
1,PERFORMANCE-Q11-720C4D8478,Q11,performance,0.0753,CAUTION
2,ABSENCE-Q01_LOW-E32C7224D6,Q01,absence,1.8051,PASSED_CHECKS
3,ABSENCE-Q05-09156E405F,Q05,absence,0.8763,PASSED_CHECKS
4,PERFORMANCE-Q01-51DF604681,Q01,performance,0.1261,PASSED_CHECKS
5,TURNOVER-Q04-C407EA67FC,Q04,turnover,0.6913,PASSED_CHECKS
6,TURNOVER-Q05-630262DAD3,Q05,turnover,0.8413,PASSED_CHECKS
7,TURNOVER-Q12-FE4D3B313A,Q12,turnover,0.8396,PASSED_CHECKS
8,PERFORMANCE-Q07-AA3EC77A5B,Q07,performance,0.0967,WITHIN_GROUP


In [16]:
# PROCESS 16 – LLM-SAFE INPUT (generic, rounded, plain-language labels)


def find_payload(item, outcome, finding_type="linear"):
    for p in structured_findings:
        if p["item"] == item and p["outcome"] == outcome and p["finding_type"] == finding_type:
            return p
    raise KeyError(f"No reportable {finding_type} finding for {item} -> {outcome}")


def build_llm_input(item, outcome, finding_type="linear"):
    p = find_payload(item, outcome, finding_type)
    ci = p["confidence_interval"]
    return {
        "item": p["item"],
        "item_label": p["item_label"],
        "finding_type": "below-3 threshold" if finding_type == "threshold" else "per one-point higher score",
        "outcome": p["outcome"],
        "method": p["method"],
        "effect_type": p["effect_type"],
        "effect": round(p["effect"], 2),
        "confidence_interval_95": [round(ci["low"], 2), round(ci["high"], 2)],
        "sample_size": p["sample_size"],
        "sample_description": f"wave 1 survey respondents ({p['sample_size']:,} of {len(w1_all):,} employees)",
        "controls": [CONTROL_LABELS.get(c, c) for c in p["controls"]] + [p["also_adjusted_for"]],
        "business_metrics": p["business_metrics"],
        "status": STATUS_LABELS[p["status"]],
        "can_be_prioritized": p["priority_eligible"],
        "status_reason": p["status_reason"],
        "notes": p["notes"],
        "required_caveats": [c["detail"] for c in p["health"]["caveats"]],
        "calculation_id": p["calculation_id"],
    }


q04_llm_result = build_llm_input("Q04", "turnover")
print(json.dumps(q04_llm_result, indent=2))

{
  "item": "Q04",
  "item_label": "Recognition",
  "finding_type": "per one-point higher score",
  "outcome": "turnover",
  "method": "Logistic Regression",
  "effect_type": "Odds Ratio",
  "effect": 0.69,
  "confidence_interval_95": [
    0.61,
    0.78
  ],
  "sample_size": 4024,
  "sample_description": "wave 1 survey respondents (4,024 of 5,000 employees)",
  "controls": [
    "tenure",
    "department",
    "role level",
    "pay band",
    "the other 11 Q12 items"
  ],
  "business_metrics": {
    "relative_odds_change_pct": -30.9,
    "relative_odds_change_ci_pct": [
      -38.6,
      -22.2
    ],
    "interpretation_unit": "odds, not probability",
    "baseline_turnover_rate_pct": 13.1,
    "illustration": {
      "turnover_rate_at_average_pct": 13.1,
      "turnover_rate_one_point_higher_pct": 9.4,
      "note": "Illustration for an otherwise average survey respondent; computed from the odds ratio."
    }
  },
  "status": "passed all of the agent's checks for common analysis t

## Evaluation against the answer key (graders only)

The cells below read `ground_truth.json`. **Nothing from the answer key is ever passed to the agent or the language model.** The scorecard fills the Pending cells in Tables 9 and 10 of the deck.

In [17]:
# PROCESS 17 – SCORECARD: TRUE EFFECTS, NULLS, TRAPS

if not ANSWER_KEY_PATH.exists():
    print("Answer key not found; skipping the scorecard.")
else:
    key = json.load(open(ANSWER_KEY_PATH))
    realized = key["realized_values"]["respondents_only"]

    def row_for(item, outcome, kind="linear"):
        m = results[(results["item"] == item) & (results["outcome"] == outcome) & (results["finding_type"] == kind)]
        return m.iloc[0] if len(m) else None

    def status_for(item, outcome, kind="linear"):
        m = final_findings[(final_findings["item"] == item) & (final_findings["outcome"] == outcome)
                           & (final_findings["finding_type"] == kind)]
        return m.iloc[0]["status"] if len(m) else "NOT_REPORTED"

    # True effects
    t_rows = []
    for t in key["true_effects"]:
        r = row_for(t["item"], t["outcome"])
        direction_ok = (r["effect"] < (1 if is_ratio(t["outcome"]) else 0)) == (t["direction"] == "negative")
        rk = next(k for k in realized if k.startswith(t["id"] + "_"))
        truth = realized[rk]
        t_rows.append({
            "id": t["id"], "item": t["item"], "outcome": t["outcome"],
            "estimate": round(r["effect"], 3), "ci": f"{r['ci_low']:.3f} to {r['ci_high']:.3f}",
            "realized_truth": truth,
            "found": bool(r["reportable"] and direction_ok),
            "status": status_for(t["item"], t["outcome"]),
            "truth_in_ci": bool(r["ci_low"] <= truth <= r["ci_high"]),
        })
    t_table = pd.DataFrame(t_rows)

    # True nulls
    n_rows = []
    for n in key["true_nulls"]:
        for o in n["outcomes"]:
            r = row_for(n["item"], o)
            n_rows.append({"id": n["id"], "item": n["item"], "outcome": o,
                           "estimate": round(r["effect"], 3), "reported": bool(r["reportable"])})
    n_table = pd.DataFrame(n_rows)

    # Traps (each graded on the agent's own outputs)
    def flag(term, outcome, check):
        d = diagnostics.get((term, outcome))
        return bool(d and d.get(check, {}).get("flag"))

    sg = diagnostics.get(("Q04", "turnover"), {}).get("subgroup", {})
    turnover_payloads = [p for p in structured_findings if p["outcome"] == "turnover"]
    traps = {
        "X1 Confounding": not bool(row_for("Q02", "turnover")["reportable"]),
        "X2 Reverse causality": status_for("Q11", "performance") in ("CAUTION", "NOT_REPORTED"),
        "X3 Hidden confounder": status_for("Q07", "absence") in ("CAUTION", "NOT_REPORTED"),
        "X4 Threshold": "Q01_low" in THRESHOLD_TERMS["absence"],
        "X5 Subgroup": bool(sg.get("flag")) and sg["new_hires"]["effect"] < sg["others"]["effect"],
        "X6 Simpson's paradox": status_for("Q07", "performance") == "WITHIN_GROUP"
                                 and row_for("Q07", "performance")["effect"] > 0,
        "X7 Nonresponse": len(turnover_payloads) > 0 and all(
            any(c["code"] == "NR_LEAVERS" for c in p["health"]["caveats"]) for p in turnover_payloads),
        "X8 Small groups": not team_view.loc[team_view["suppressed"], ["Q04", "left_voluntarily_12m"]].notna().any().any()
                           and int(team_view["suppressed"].sum()) >= int(key["dataset"]["small_teams"]),
    }
    trap_table = pd.DataFrame([{"trap": k, "handled": v} for k, v in traps.items()])

    # Findings that are neither true effects nor known traps
    known = {(t["item"], t["outcome"]) for t in key["true_effects"]} | {
        ("Q11", "performance"), ("Q07", "absence"), ("Q07", "performance")}
    unexpected = final_findings[[(r["item"], r["outcome"]) not in known and r["finding_type"] == "linear"
                                 for _, r in final_findings.iterrows()]]

    print(f"True effects found: {int(t_table['found'].sum())} of {len(t_table)}")
    print(f"Truth inside the reported 95% CI: {int(t_table['truth_in_ci'].sum())} of {len(t_table)}")
    print(f"Nulls reported as effects: {int(n_table['reported'].sum())} of {len(n_table)}")
    print(f"Traps handled: {int(trap_table['handled'].sum())} of 8 (pass bar 6)")
    print(f"Unexpected reportable findings: {len(unexpected)}")
    display(t_table)
    display(n_table)
    display(trap_table)
    if len(unexpected):
        display(unexpected[["item", "outcome", "effect", "status"]])

True effects found: 5 of 5
Truth inside the reported 95% CI: 5 of 5
Nulls reported as effects: 0 of 7
Traps handled: 8 of 8 (pass bar 6)
Unexpected reportable findings: 0


,id,item,outcome,estimate,ci,realized_truth,found,status,truth_in_ci
0,T1,Q04,turnover,0.691,0.614 to 0.778,0.691,True,PASSED_CHECKS,True
1,T2,Q05,turnover,0.841,0.751 to 0.942,0.841,True,PASSED_CHECKS,True
2,T3,Q12,turnover,0.840,0.752 to 0.938,0.840,True,PASSED_CHECKS,True
3,T4,Q01,performance,0.126,0.096 to 0.156,0.126,True,PASSED_CHECKS,True
4,T5,Q05,absence,0.876,0.843 to 0.911,0.876,True,PASSED_CHECKS,True


,id,item,outcome,estimate,reported
0,N1,Q06,absence,0.984,False
1,N2,Q10,turnover,1.045,False
2,N2,Q10,absence,0.994,False
3,N2,Q10,performance,-0.012,False
4,N3,Q08,turnover,0.983,False
5,N3,Q08,absence,1.010,False
6,N3,Q08,performance,0.006,False


,trap,handled
0,X1 Confounding,True
1,X2 Reverse causality,True
2,X3 Hidden confounder,True
3,X4 Threshold,True
4,X5 Subgroup,True
5,X6 Simpson's paradox,True
6,X7 Nonresponse,True
7,X8 Small groups,True


## Language model: explain verified results

Runs only when `RUN_LLM=1` (default) and an Azure key is available. Every answer is checked automatically before it is shown.

In [18]:
# PROCESS 18 – CONNECT TO GW AZURE OPENAI (optional)

RUN_LLM = os.getenv("RUN_LLM", "1") == "1"
MODEL = os.getenv("GALLUP_LLM_MODEL", "gpt-5-nano")
client = None

if RUN_LLM:
    from getpass import getpass
    from openai import AzureOpenAI

    RESOURCE = os.getenv("GW_AZURE_RESOURCE", "gw-sb-10")
    api_key = os.getenv("GW_AZURE_OPENAI_KEY") or getpass("Azure OpenAI API key: ")
    client = AzureOpenAI(
        azure_endpoint=f"https://{RESOURCE}.openai.azure.com/",
        api_key=api_key,
        api_version="2024-12-01-preview",
    )
    print("Azure OpenAI client ready. Model:", MODEL)
else:
    print("RUN_LLM=0: skipping the language model cells.")

Azure OpenAI client ready. Model: gpt-5-nano


In [19]:
# PROCESS 19 – SYSTEM PROMPT

SYSTEM_PROMPT = """
You are an engagement decision-support agent. You explain verified statistical
results to nontechnical business users (managers, HR, finance and operations leaders).

The numbers were computed by deterministic Python tools. You only interpret them.

RULES
1. Use ONLY the numbers in VERIFIED_RESULT. Never invent, estimate, recalculate or
   round differently than supplied.
2. Never claim causation. Say "associated with" or "linked with"; never "causes",
   "drives", "leads to", "results in" or "impacts".
3. Open with one plain sentence that states the direction, for example
   "Employees with higher Recognition scores are less likely to leave."
4. Give the effect size, its 95% confidence interval and the sample description once.
5. Say that the result already accounts for the listed controls, including the other
   Q12 items.
6. Odds are not probabilities. Describe an odds ratio with the supplied percentage
   change in odds. If an illustration in business_metrics is supplied, use it to say
   what the change means in plain terms, and say it is an illustration.
7. If notes are supplied (for example a subgroup or threshold pattern), include them.
8. Include every required caveat, in plain words.
9. Status:
   - If can_be_prioritized is false, lead with the verdict: say the finding should not
     currently be prioritized, then explain why using status_reason.
   - If the status says the result holds within departments, explain that the
     company-wide pattern points the other way and that the within-department result
     is the one to use.
10. Do not recommend actions unless the user asks for recommendations or next steps.
11. Never discuss individual employees.
12. Do not show p-values, field names, codes or true/false values.
13. Keep it short: a headline, 3 to 5 short bullets, then caveats.
14. End with "Calculation ID: <id>" exactly once.
"""

CAUSAL_WORDS = ["causes", "caused by", "drives", "driven by", "leads to", "results in", "impacts"]
print("System prompt ready.")

System prompt ready.


In [20]:
# PROCESS 20 – EXPLAIN A FINDING AND CHECK THE ANSWER

import re


def allowed_numbers(obj):
    """Every number in the payload, plus common display variants (rounded, percent, absolute)."""
    text = re.sub(r"(?<=\d),(?=\d{3})", "", json.dumps(obj))   # 4,024 -> 4024
    nums = set()
    for s in re.findall(r"-?\d+(?:\.\d+)?", text):
        v = float(s)
        for x in {v, abs(v), v * 100, abs(v) * 100, 100 - abs(v) * 100, abs(v * 100) - 100, abs(100 - v * 100)}:
            nums.update({f"{x:.0f}", f"{x:.1f}", f"{x:.2f}", f"{x:.3f}"})
    nums.update(str(i) for i in range(0, 13))   # item numbers, small counts
    nums.update({"95", "100"})
    return nums


def check_answer(answer, verified):
    issues = []
    low = answer.lower()
    for w in CAUSAL_WORDS:
        if re.search(rf"\b{w}\b", low):
            issues.append(f"causal wording: '{w}'")
    if answer.count(verified["calculation_id"]) != 1:
        issues.append("calculation ID missing or repeated")
    ok = allowed_numbers(verified)
    found = [n.replace(",", "") for n in re.findall(r"\d[\d,]*(?:\.\d+)?", answer)]
    ungrounded = sorted({n for n in found if n not in ok and n.lstrip("0") not in ok
                         and n not in verified["calculation_id"]})
    if ungrounded:
        issues.append(f"numbers not in the payload: {ungrounded}")
    return issues


def explain(item, outcome, question, finding_type="linear"):
    verified = build_llm_input(item, outcome, finding_type)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"USER QUESTION:\n{question}\n\nVERIFIED_RESULT:\n{json.dumps(verified, indent=2)}"},
    ]
    try:
        response = client.chat.completions.create(model=MODEL, messages=messages,
                                                  max_completion_tokens=2000, reasoning_effort="low")
    except Exception as e:  # older API versions may not accept reasoning_effort
        print("Retrying without reasoning_effort:", type(e).__name__)
        response = client.chat.completions.create(model=MODEL, messages=messages, max_completion_tokens=3000)
    answer = response.choices[0].message.content or ""
    issues = check_answer(answer, verified)
    return answer, issues, response


print("explain() ready.")

explain() ready.


In [21]:
# PROCESS 21 – TEST: Q04 RECOGNITION AND TURNOVER

if RUN_LLM:
    answer_q04, issues_q04, response_q04 = explain(
        "Q04", "turnover",
        "What does Q04 Recognition tell us about voluntary turnover? Explain the finding and what it means for the business.",
    )
    print(answer_q04)
    print("\nAutomatic check:", "passed" if not issues_q04 else issues_q04)

Recognition is associated with lower voluntary turnover.

- Effect: Each one-point higher Q04 Recognition score is linked with a 0.69 odds ratio for turnover (i.e., about a 30.9% reduction in odds) with a 95% CI of 0.61 to 0.78.
- Sample and controls: Based on 4,024 wave 1 respondents, controlling for tenure, department, role level, pay band, and the other 11 Q12 items.
- Interpretation: The business illustration shows an average baseline turnover rate of 13.1%; with a one-point higher Recognition score, turnover is illustrated at 9.4% for an otherwise average respondent. This is an illustration of the change in odds, not a direct probability change.
- Notes: The association differs by tenure: for employees with under 2 years, the effect is stronger (0.52; 95% CI 0.43 to 0.64, n = 1,316) than for others (0.82; 95% CI 0.70 to 0.95, n = 2,708). The result already accounts for the listed controls, including the other Q12 items.

Caveats:
- The survey response rate was 57.7% among employee

In [22]:
# PROCESS 22 – TEST: Q07 OPINIONS COUNT AND ABSENCE (a finding that should not be prioritized)

if RUN_LLM:
    try:
        answer_q07, issues_q07, response_q07 = explain(
            "Q07", "absence",
            "Is Q07 Opinions Count an important engagement factor for absenteeism? "
            "Explain the finding, its limitations, and whether it should be prioritized.",
        )
        print(answer_q07)
        print("\nAutomatic check:", "passed" if not issues_q07 else issues_q07)
    except KeyError as e:
        print(e)

Q07 Opinions Count is associated with absenteeism: higher scores are linked with fewer absence days.

- Effect size: incidence rate ratio (IRR) 0.81 per one-point higher Q07 score (95% CI 0.78 to 0.84), corresponding to about a 18.8% decrease in the rate of absence (CI -21.7% to -15.7%). The result already accounts for the listed controls, including the other Q12 items.
- Illustration: at average absence of 3.2 days, a one-point higher Q07 score is associated with about 2.6 days of absence.
- Sample: 4,024 wave 1 survey respondents (out of 5,000 employees); controls include tenure, department, work shift, and the other 11 Q12 items.

 caveats and status
- Status: did not pass the agent's checks, so it should not be prioritized. The within-team pattern shrank notably when comparing employees within the same team, suggesting a team/manager factor explains much of the pattern.
- Notes: survey nonresponse may bias absence estimates (standardized difference -0.20); results differ across dep

In [23]:
# PROCESS 23 – USAGE AND LATENCY

if RUN_LLM:
    for name, resp in [("Q04", response_q04), ("Q07", globals().get("response_q07"))]:
        if resp is None:
            continue
        u = resp.usage
        details = getattr(u, "completion_tokens_details", None)
        reasoning = getattr(details, "reasoning_tokens", None) if details else None
        print(f"{name}: finish={resp.choices[0].finish_reason}, prompt={u.prompt_tokens}, "
              f"completion={u.completion_tokens}, reasoning={reasoning}")

Q04: finish=stop, prompt=906, completion=564, reasoning=256
Q07: finish=stop, prompt=894, completion=656, reasoning=384
